# 🌋 MintPy Interactive Analysis Notebook

This notebook provides an interactive environment for exploring MintPy InSAR time-series results.

**Dataset:** Fernandina Volcano, Galápagos Islands (Sentinel-1, Track 128)

**Prerequisites:** Run the pipeline scripts first (`01_download_data.sh` and `02_run_pipeline.sh`).

## 1. Setup & Imports

In [ ]:
import os
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt
from datetime import datetime
import glob

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Dataset configuration
DATASET = 'FernandinaSenDT128'
WORK_DIR = Path(os.environ.get('MINTPY_WORK_DIR', f'/data/{DATASET}/mintpy'))
if not WORK_DIR.exists():
    example_dirs = [
        Path('/mnt/data/aoi_3_bologna/mintpy_filtered'),
        Path('/mnt/data/aoi_3_bologna/mintpy'),
    ]
    available = [str(path) for path in example_dirs if path.exists()]
    raise FileNotFoundError(
        f'{WORK_DIR} does not exist. Set MINTPY_WORK_DIR to a MintPy output directory or edit WORK_DIR. '
        f'Available examples: {available or "none found"}'
    )
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Available files:')
for f in sorted(glob.glob('*.h5')):
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {f:40s} ({size_mb:.1f} MB)')

## 2. Explore the Interferogram Stack

In [ ]:
# Load interferogram stack metadata
with h5py.File('inputs/ifgramStack.h5', 'r') as f:
    print('=== ifgramStack.h5 Structure ===')
    print(f'Datasets: {list(f.keys())}')
    print(f'\nDate pairs: {f["date"].shape[0]}')
    print(f'First 5 date pairs:')
    for d in f['date'][:5]:
        print(f'  {d.decode()}')
    
    if 'unwrapPhase' in f:
        print(f'\nUnwrapped phase shape: {f["unwrapPhase"].shape}')
    if 'coherence' in f:
        print(f'Coherence shape: {f["coherence"].shape}')
    if 'dropIfgram' in f:
        n_drop = np.sum(~f['dropIfgram'][:])
        print(f'Dropped interferograms: {n_drop}')
    
    print(f'\nAttributes:')
    for key in sorted(f.attrs.keys())[:15]:
        print(f'  {key}: {f.attrs[key]}')

## 3. View Velocity Map

In [ ]:
with h5py.File('velocity.h5', 'r') as f:
    vel = f['velocity'][:] * 100  # m/yr → cm/yr

vel[vel == 0] = np.nan
vmax = np.nanpercentile(np.abs(vel), 95)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(vel, cmap='jet', vmin=-vmax, vmax=vmax)
cbar = fig.colorbar(im, ax=ax, shrink=0.7)
cbar.set_label('LOS Velocity (cm/yr)')
ax.set_title(f'LOS Velocity — {DATASET}', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Temporal Coherence

In [ ]:
with h5py.File('temporalCoherence.h5', 'r') as f:
    coh = list(f.values())[0][:]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(coh, cmap='gray', vmin=0, vmax=1)
fig.colorbar(im, ax=axes[0], shrink=0.7)
axes[0].set_title('Temporal Coherence Map')

axes[1].hist(coh.flatten(), bins=100, color='steelblue', edgecolor='white')
axes[1].axvline(0.7, color='red', linestyle='--', label='Typical threshold (0.7)')
axes[1].set_xlabel('Temporal Coherence')
axes[1].set_ylabel('Pixel Count')
axes[1].set_title('Coherence Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Interactive Time-Series at a Point

Click on the velocity map to select a point and see its time-series.

In [ ]:
# Load the time-series
ts_files = sorted(glob.glob('timeseries*.h5'))
ts_file = ts_files[-1]  # Most corrected version
print(f'Using: {ts_file}')

with h5py.File(ts_file, 'r') as f:
    ts_data = f['timeseries'][:]
    dates = [d.decode() for d in f['date'][:]]

date_dts = [datetime.strptime(d, '%Y%m%d') for d in dates]

# Plot time-series at a few selected points
ntime, nrow, ncol = ts_data.shape
points = {
    'Center': (nrow//2, ncol//2),
    'NW quadrant': (nrow//4, ncol//4),
    'SE quadrant': (3*nrow//4, 3*ncol//4),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: velocity map with marked points
axes[0].imshow(vel, cmap='jet', vmin=-vmax, vmax=vmax)
colors = ['red', 'blue', 'green']
for (label, (r, c)), color in zip(points.items(), colors):
    axes[0].plot(c, r, 'o', color=color, markersize=10, markeredgecolor='white', markeredgewidth=2)
axes[0].set_title('Selected Points')

# Right: time-series
for (label, (r, c)), color in zip(points.items(), colors):
    disp = ts_data[:, r, c] * 100  # to cm
    axes[1].plot(date_dts, disp, 'o-', label=label, color=color, markersize=3)

axes[1].set_xlabel('Date')
axes[1].set_ylabel('LOS Displacement (cm)')
axes[1].set_title('Displacement Time-Series')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Geometry Files

In [ ]:
geometry_file = 'inputs/geometryRadar.h5' if Path('inputs/geometryRadar.h5').exists() else 'inputs/geometryGeo.h5'

with h5py.File(geometry_file, 'r') as f:
    print(f'Geometry datasets: {list(f.keys())}')
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    datasets = ['height', 'incidenceAngle', 'shadowMask', 'waterMask']
    cmaps = ['terrain', 'viridis', 'gray', 'Blues']
    
    for ax, ds_name, cmap in zip(axes.flat, datasets, cmaps):
        if ds_name in f:
            data = f[ds_name][:]
            im = ax.imshow(data, cmap=cmap)
            fig.colorbar(im, ax=ax, shrink=0.7)
            ax.set_title(ds_name)
        else:
            ax.set_title(f'{ds_name} (not available)')
    
    plt.suptitle('Geometry Files', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.show()

## 7. Using MintPy CLI from the Notebook

In [ ]:
# You can run any MintPy command directly from the notebook
!info.py velocity.h5

In [ ]:
# Check all available MintPy tools
!smallbaselineApp.py --help 2>&1 | head -30

## 8. Using MintPy Python API

In [ ]:
from mintpy.utils import readfile

# Read velocity data using MintPy's API
vel_data, vel_meta = readfile.read('velocity.h5')
print(f'Shape: {vel_data.shape}')
print(f'\nMetadata:')
for key in ['LENGTH', 'WIDTH', 'UNIT', 'REF_Y', 'REF_X', 'REF_LAT', 'REF_LON']:
    if key in vel_meta:
        print(f'  {key}: {vel_meta[key]}')